# DTD Jupyter 计算模块

这个 notebook 将已确认的 `vanke` 历史表与尚未确认的 `temporary` 表合并，并严格使用目标日期及以前的数据计算一个 DTD 数值。

- confirmed 与 temporary 使用相同字段；若公司和日期重复，confirmed 优先。
- temporary 中的新交易日会接到历史数据之后；缺失的财务报表字段会从之前最近一期向前带入。
- 目标日期必须真实存在，不会自动偷用前一个交易日。
- 默认使用一年滚动窗口、每年 250 个交易日、至少 50 个有效观测和 δ = 0.50。
- 这是按提供说明重建的 NUS CRI 方法一致版本，不声称精确复刻专有生产参数。

## 1 输入参数

日常使用时只需修改本格：

1. `CONFIRMED_FILE`：已确认的历史文件。
2. `TEMPORARY_FILE`：未确认文件；没有时填 `None`。也可以直接传入一个 pandas DataFrame。
3. `TARGET_DATE`：需要计算的交易日。
4. 如果文件中有多家公司，再填写 `COMPANY`。

In [1]:
from pathlib import Path

CONFIRMED_FILE = Path(r"C:\Users\susul\Desktop\苏苏重要文件备份\NUS CRI\vanke.xlsx")
TEMPORARY_FILE = None  # 例如 Path(r"C:\data\temporary.xlsx")
TARGET_DATE = "2025-12-12"
COMPANY = None         # 单一公司可保持 None；多公司时填 Comp_no，例如 5338
INPUT_SHEET = "Input" # xlsx 文件中的输入工作表名称

## 2 完整计算模块

In [2]:
from __future__ import annotations

from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Optional, Union

import numpy as np
import pandas as pd
from scipy.optimize import brentq, minimize_scalar
from scipy.stats import norm


COLS = {
    "company": "Comp_no",
    "date": "Date",
    "equity": "CUR_MKT_CAP(HKD)",
    "cl": "BS_CUR_LIAB(HKD)",
    "ltd": "BS_LT_BORROW(HKD)",
    "tl": "BS_TOT_LIAB2(HKD)",
    "assets": "BS_TOT_ASSET(HKD)",
    "rf": "Risk_Free_Rate",
}
REQUIRED_COLUMNS = [COLS[key] for key in ("date", "equity", "cl", "ltd", "tl", "assets", "rf")]
BS_COLUMNS = [COLS[key] for key in ("cl", "ltd", "tl", "assets")]
TableInput = Union[str, Path, pd.DataFrame]


@dataclass(frozen=True)
class DTDResult:
    date: pd.Timestamp
    dtd: float
    sigma: float
    asset_value: float
    default_point: float
    n_obs: int
    delta: float
    source_status: str

    def as_dict(self) -> dict:
        result = asdict(self)
        result["date"] = self.date.strftime("%Y-%m-%d")
        return result


def _parse_dates(series: pd.Series) -> pd.Series:
    """同时接受 Excel 日期、YYYYMMDD 数字和普通日期字符串。"""
    text = series.astype(str).str.strip().str.replace(r"\.0$", "", regex=True)
    compact = text.str.fullmatch(r"\d{8}")
    parsed = pd.to_datetime(text, errors="coerce")
    if compact.any():
        parsed.loc[compact] = pd.to_datetime(text.loc[compact], format="%Y%m%d", errors="coerce")
    return parsed.dt.normalize()


def _rate_to_decimal(value: float) -> float:
    """2.46 解释为 2.46%；0.0246 保持为小数利率。"""
    value = float(value)
    return value / 100.0 if abs(value) > 1.0 else value


def _read_table(source: Optional[TableInput], sheet_name: str = "Input") -> pd.DataFrame:
    if source is None:
        return pd.DataFrame()
    if isinstance(source, pd.DataFrame):
        return source.copy()
    path = Path(source)
    if not path.exists():
        raise FileNotFoundError(f"找不到输入文件: {path}")
    suffix = path.suffix.lower()
    if suffix in {".xlsx", ".xls", ".xlsm"}:
        return pd.read_excel(path, sheet_name=sheet_name)
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix == ".parquet":
        return pd.read_parquet(path)
    raise ValueError(f"不支持的文件格式: {suffix}")


def _validate_schema(frame: pd.DataFrame, label: str) -> None:
    missing = [column for column in REQUIRED_COLUMNS if column not in frame.columns]
    if missing:
        raise ValueError(f"{label} 缺少字段: {missing}")


def combine_confirmed_and_temporary(
    confirmed: TableInput,
    temporary: Optional[TableInput] = None,
    *,
    confirmed_sheet: str = "Input",
    temporary_sheet: str = "Input",
    company: Optional[object] = None,
) -> pd.DataFrame:
    """合并数据并返回干净历史表；重复日期时 confirmed 覆盖 temporary。"""
    confirmed_df = _read_table(confirmed, confirmed_sheet)
    temporary_df = _read_table(temporary, temporary_sheet)
    _validate_schema(confirmed_df, "confirmed")
    if not temporary_df.empty:
        _validate_schema(temporary_df, "temporary")

    confirmed_df = confirmed_df.copy()
    confirmed_df["_source_status"] = "confirmed"

    frames = [confirmed_df]
    if not temporary_df.empty:
        temporary_df = temporary_df.copy()
        temporary_df["_source_status"] = "temporary"
        # temporary 若没有 Comp_no，而 confirmed 只有一家公司，则自动补齐。
        if COLS["company"] not in temporary_df.columns and COLS["company"] in confirmed_df.columns:
            companies = confirmed_df[COLS["company"]].dropna().unique()
            if len(companies) == 1:
                temporary_df[COLS["company"]] = companies[0]
        frames.append(temporary_df)

    data = pd.concat(frames, ignore_index=True, sort=False)
    data[COLS["date"]] = _parse_dates(data[COLS["date"]])
    if data[COLS["date"]].isna().any():
        bad_rows = data.index[data[COLS["date"]].isna()].tolist()[:10]
        raise ValueError(f"存在无法识别的日期，行号示例: {bad_rows}")

    has_company = COLS["company"] in data.columns
    if company is not None:
        if not has_company:
            raise ValueError("指定了 company，但输入中没有 Comp_no 字段")
        data = data[data[COLS["company"]] == company].copy()
        if data.empty:
            raise ValueError(f"找不到 Comp_no={company} 的数据")
    elif has_company:
        companies = data[COLS["company"]].dropna().unique()
        if len(companies) > 1:
            raise ValueError(f"文件含多家公司 {companies.tolist()}，请设置 COMPANY")

    data["_source_priority"] = data["_source_status"].map({"temporary": 1, "confirmed": 2})
    key_columns = ([COLS["company"]] if has_company else []) + [COLS["date"]]
    data = (
        data.sort_values(key_columns + ["_source_priority"])
        .drop_duplicates(key_columns, keep="last")
        .sort_values(key_columns)
        .reset_index(drop=True)
    )

    numeric_columns = [COLS[key] for key in ("equity", "cl", "ltd", "tl", "assets", "rf")]
    for column in numeric_columns:
        data[column] = pd.to_numeric(data[column], errors="coerce")

    # 财报字段不是每日更新；temporary 新行缺失时从最近已知财报向前带入。
    if has_company:
        data[BS_COLUMNS] = data.groupby(COLS["company"], dropna=False)[BS_COLUMNS].ffill()
    else:
        data[BS_COLUMNS] = data[BS_COLUMNS].ffill()

    if data[BS_COLUMNS].isna().any().any():
        raise ValueError("财报字段无法向前带入；请确保历史首行有完整财报数据")
    return data.drop(columns="_source_priority")


def add_model_fields(frame: pd.DataFrame, delta: float = 0.5) -> pd.DataFrame:
    if not 0.0 <= delta <= 1.0:
        raise ValueError("delta 必须在 0 到 1 之间")
    data = frame.copy()
    data[COLS["date"]] = _parse_dates(data[COLS["date"]])
    data = data.sort_values(COLS["date"]).reset_index(drop=True)
    data["Other_Liabilities(HKD)"] = data[COLS["tl"]] - data[COLS["cl"]] - data[COLS["ltd"]]
    data["Default_Point(HKD)"] = (
        data[COLS["cl"]] + 0.5 * data[COLS["ltd"]] + delta * data["Other_Liabilities(HKD)"]
    )
    data["Risk_Free_Decimal"] = data[COLS["rf"]].map(_rate_to_decimal)
    return data


def implied_asset_value(
    equity: float,
    debt: float,
    rate: float,
    sigma: float,
    maturity: float = 1.0,
) -> float:
    """由 E = V N(d1) - exp(-rT)L N(d2) 反解市场资产价值 V。"""
    equity, debt, rate, sigma, maturity = map(float, (equity, debt, rate, sigma, maturity))
    if equity <= 0 or debt <= 0 or sigma <= 0 or maturity <= 0:
        raise ValueError("equity、debt、sigma 和 maturity 必须为正")
    sqrt_t = np.sqrt(maturity)

    def residual(asset_value: float) -> float:
        d1 = (
            np.log(asset_value / debt) + (rate + 0.5 * sigma**2) * maturity
        ) / (sigma * sqrt_t)
        d2 = d1 - sigma * sqrt_t
        option_value = asset_value * norm.cdf(d1) - debt * np.exp(-rate * maturity) * norm.cdf(d2)
        return option_value - equity

    lower = max(equity, 1e-9)
    upper = equity + debt * np.exp(-rate * maturity) + debt
    while residual(upper) < 0:
        upper *= 2.0
        if upper > 1e12 * max(1.0, debt):
            raise RuntimeError("无法为市场资产价值 V 建立求根区间")
    return float(brentq(residual, lower, upper, maxiter=200))


def _window_valid_rows(data: pd.DataFrame, as_of: pd.Timestamp, lookback_days: int) -> pd.DataFrame:
    start = as_of - pd.Timedelta(days=lookback_days)
    window = data[(data[COLS["date"]] > start) & (data[COLS["date"]] <= as_of)].copy()
    needed = REQUIRED_COLUMNS + ["Default_Point(HKD)", "Risk_Free_Decimal"]
    window = window.dropna(subset=needed)
    positive = (
        (window[COLS["equity"]] > 0)
        & (window[COLS["assets"]] > 0)
        & (window["Default_Point(HKD)"] > 0)
    )
    return window.loc[positive].reset_index(drop=True)


def estimate_sigma(
    history: pd.DataFrame,
    as_of,
    *,
    delta: float = 0.5,
    lookback_days: int = 365,
    min_obs: int = 50,
    trading_days_per_year: int = 250,
    sigma_bounds: tuple[float, float] = (0.005, 1.50),
) -> tuple[float, pd.DataFrame]:
    """按一年度滚动 transformed-data MLE 估计资产波动率 sigma。"""
    as_of = pd.Timestamp(as_of).normalize()
    data = add_model_fields(history, delta)
    window = _window_valid_rows(data, as_of, lookback_days)
    if len(window) < min_obs:
        raise ValueError(f"至少需要 {min_obs} 个有效观测；当前只有 {len(window)} 个")

    equity = window[COLS["equity"]].to_numpy(float)
    debt = window["Default_Point(HKD)"].to_numpy(float)
    book_assets = window[COLS["assets"]].to_numpy(float)
    rate = window["Risk_Free_Decimal"].to_numpy(float)
    h = np.full(len(window) - 1, 1.0 / trading_days_per_year)

    def negative_log_likelihood(sigma: float) -> float:
        try:
            market_assets = np.array(
                [implied_asset_value(equity[i], debt[i], rate[i], sigma) for i in range(len(window))]
            )
            d1 = (np.log(market_assets / debt) + rate + 0.5 * sigma**2) / sigma
            normal_d1 = np.clip(norm.cdf(d1), 1e-14, 1.0)
            transformed_return = np.log(
                (market_assets[1:] / book_assets[1:]) * (book_assets[:-1] / market_assets[:-1])
            )
            mu = 0.5 * sigma**2 + transformed_return.sum() / h.sum()
            residual = transformed_return - (mu - 0.5 * sigma**2) * h
            log_likelihood = (
                -0.5 * (len(window) - 1) * np.log(2 * np.pi)
                -0.5 * np.sum(np.log(sigma**2 * h))
                -np.sum(np.log(market_assets[1:] / book_assets[1:]))
                -np.sum(np.log(normal_d1[1:]))
                -0.5 / sigma**2 * np.sum((residual**2) / h)
            )
            return -float(log_likelihood) if np.isfinite(log_likelihood) else 1e100
        except (ValueError, RuntimeError, FloatingPointError):
            return 1e100

    optimum = minimize_scalar(
        negative_log_likelihood,
        bounds=sigma_bounds,
        method="bounded",
        options={"xatol": 1e-8},
    )
    if not optimum.success or not np.isfinite(optimum.fun):
        raise RuntimeError("sigma 优化未收敛")
    return float(optimum.x), window


def calculate_dtd(
    history: pd.DataFrame,
    as_of,
    *,
    delta: float = 0.5,
    lookback_days: int = 365,
    min_obs: int = 50,
    trading_days_per_year: int = 250,
) -> DTDResult:
    """只使用 as_of 当天及以前的数据，计算该交易日 DTD。"""
    as_of = pd.Timestamp(as_of).normalize()
    data = add_model_fields(history, delta)
    available_dates = data[COLS["date"]]
    if not (available_dates == as_of).any():
        latest = available_dates[available_dates <= as_of].max()
        latest_text = "无" if pd.isna(latest) else latest.strftime("%Y-%m-%d")
        raise ValueError(f"目标日期 {as_of.date()} 不在输入中；不晚于目标日的最近日期是 {latest_text}")
    data = data[data[COLS["date"]] <= as_of].copy()
    row = data.loc[data[COLS["date"]] == as_of].iloc[-1]

    sigma, window = estimate_sigma(
        data,
        as_of,
        delta=delta,
        lookback_days=lookback_days,
        min_obs=min_obs,
        trading_days_per_year=trading_days_per_year,
    )
    market_assets = implied_asset_value(
        row[COLS["equity"]],
        row["Default_Point(HKD)"],
        row["Risk_Free_Decimal"],
        sigma,
    )
    # 按提供说明，CRI DTD 中的漂移处理使该项抵消。
    dtd = np.log(market_assets / row["Default_Point(HKD)"]) / sigma
    return DTDResult(
        date=as_of,
        dtd=float(dtd),
        sigma=float(sigma),
        asset_value=float(market_assets),
        default_point=float(row["Default_Point(HKD)"]),
        n_obs=len(window),
        delta=float(delta),
        source_status=str(row.get("_source_status", "unknown")),
    )


def run_dtd(
    confirmed_file: TableInput,
    as_of,
    temporary_file: Optional[TableInput] = None,
    *,
    company: Optional[object] = None,
    confirmed_sheet: str = "Input",
    temporary_sheet: str = "Input",
    delta: float = 0.5,
    lookback_days: int = 365,
    min_obs: int = 50,
    trading_days_per_year: int = 250,
    return_details: bool = False,
):
    """主入口。默认只返回 float；return_details=True 时返回 DTDResult。"""
    clean_history = combine_confirmed_and_temporary(
        confirmed_file,
        temporary_file,
        confirmed_sheet=confirmed_sheet,
        temporary_sheet=temporary_sheet,
        company=company,
    )
    result = calculate_dtd(
        clean_history,
        as_of,
        delta=delta,
        lookback_days=lookback_days,
        min_obs=min_obs,
        trading_days_per_year=trading_days_per_year,
    )
    return result if return_details else result.dtd

## 3 执行并取得单一 DTD 数值

下面先显示必要诊断，再把最后一个表达式保留为纯 Python `float`。如果只需要数字，也可以直接运行：

```python
run_dtd(CONFIRMED_FILE, TARGET_DATE, TEMPORARY_FILE, company=COMPANY)
```

In [3]:
details = run_dtd(
    confirmed_file=CONFIRMED_FILE,
    temporary_file=TEMPORARY_FILE,
    as_of=TARGET_DATE,
    company=COMPANY,
    confirmed_sheet=INPUT_SHEET,
    temporary_sheet=INPUT_SHEET,
    return_details=True,
)

display(pd.Series(details.as_dict(), name="value"))
dtd_number = float(details.dtd)
dtd_number

date                2025-12-12
dtd                   1.359111
sigma                 0.038964
asset_value      826896.213194
default_point    784245.644885
n_obs                      246
delta                      0.5
source_status        confirmed
Name: value, dtype: object

1.3591112124281255